# 11_03 Extractive question answering: why did the reader answer a question it could not?

A reader is a BERT fine-tuned to point at the answer to a question inside a passage. This one is
**DistilBERT** fine-tuned on SQuAD 1.1, about 100,000 questions about Wikipedia paragraphs. You put
Kittiwake's five service notices in front of it and find out where it is excellent, and where it is
confidently wrong.

**How this notebook works.** Every notebook in this course has the same rhythm:

1. **Recall.** Answer from memory before you look anything up. `ask()` tells you at once whether you were right.
2. **Predict, then run.** Before a cell with a surprise in it, write your prediction into `guess()`. The next cell runs the code and `reveal()` compares.
3. **Worked example, then your turn.** One example is done in full; the next, near-identical one has lines marked `# YOUR CODE HERE`.
4. **Check.** A `check_...()` cell tests what you saved, exactly as the checkpoint will, and says what to fix.

Run cells in order with **Shift+Enter**. If you get lost, **Kernel, Restart Kernel and Run All Cells** starts clean.

Running this in Google Colab? This cell sets it up; in CourseLabs it does nothing.

In [ ]:
# Colab setup. In a CourseLabs session this cell does nothing.
import os, sys
if "google.colab" in sys.modules:
    import importlib, importlib.util, subprocess
    LAB, REPO = "lab-nlp-11-bert-and-what-came-after", "/content/nlp-course"
    if not os.path.isdir(REPO):
        subprocess.run(["git", "clone", "-q", "--depth", "1", "https://github.com/fenago/nlp-course.git", REPO], check=True)
    os.chdir(f"{REPO}/{LAB}")
    if not os.path.exists("data"):
        os.symlink("../data", "data")
    os.makedirs("out", exist_ok=True)
    os.environ["NLPLAB_DATA"] = f"{REPO}/data"
    sys.path.insert(0, os.getcwd())
    PIP = {'transformers': 'transformers',
           'torch': 'torch',
           'sklearn': 'scikit-learn',
           'pandas': 'pandas',
           'numpy': 'numpy'}
    missing = [spec for mod, spec in PIP.items() if importlib.util.find_spec(mod) is None]
    if missing:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", *missing], check=True)
        importlib.invalidate_caches()
    print(f"Ready: {LAB} and its data are in {os.getcwd()}; installed {len(missing)} package(s).")
elif not os.path.isdir("/opt/nlplab/data") and os.path.isdir("data"):
    # A downloaded copy on your own computer: the helpers read data/ from here.
    os.environ["NLPLAB_DATA"] = os.path.abspath("data")

In [ ]:
import json
import os
import bertlab
from nlpcheck import ask, guess, reveal, check_11_03

os.makedirs("out", exist_ok=True)
results = {}
notices = bertlab.notices()
for i, n in enumerate(notices):
    print(i, n[:110], "...")

## 1. Recall

**r5.** In 11_02, what does fine-tuning train? (a) only the new head, (b) only the pretrained layers,
(c) the new head and every pretrained weight, a little

**r6.** Why could no model in 11_02 get far above 0.92 macro-F1 on the tickets?
(a) about one ticket in twenty carries the wrong department, (b) bert-tiny is too small, (c) 450 tickets
is too few

In [ ]:
ask("r5", "")
ask("r6", "")

## 2. The worked example: one question, one notice

`read(question, context)` does what the page described. It feeds `[CLS] question [SEP] notice [SEP]` to the
reader, which gives every token a **start logit** and an **end logit**: raw scores, before any softmax.
Then it returns the span whose start logit plus end logit is highest. It also reports `prob`, the start
softmax times the end softmax, which is what Hugging Face's old question-answering pipeline called the
score.

In [ ]:
r = bertlab.read("When will the mast be upgraded?", notices[0])
results["mast"] = r["answer"]
print(r)

`14 October`, with a probability near 1. The reader wrote nothing: it pointed at two tokens of the notice,
and the answer is whatever lies between them.

## 3. A question the notice cannot answer

Notice 0 mentions Unlimited Plus but not its price. Ask it "How much is Unlimited Plus?". Predict: will the
reader say it does not know, or will it return a span? (write "no answer" or "a span")

In [ ]:
guess("unanswerable", None)

In [ ]:
r = bertlab.read("How much is Unlimited Plus?", notices[0])
results["unlimited"] = r["answer"]
print(r)
reveal("unanswerable", "no answer" if r["answer"] == "" else "a span")

A span. A reader fine-tuned on SQuAD 1.1 has no way to say "the passage does not say": every one of its
100,000 training questions had an answer in the paragraph, so it always points somewhere. SQuAD 2.0 added
50,000 unanswerable questions for exactly this reason, and readers trained on it learn to point at `[CLS]`
instead. With this reader, deciding not to answer is your job.

## 4. Which notice?

A customer does not tell you which notice to read. The obvious approach is to run the reader over all five
and keep the answer it is most sure of. Predict what it will answer to "How much will Flex 30 cost?" when
the answer with the highest `prob` wins.

In [ ]:
guess("flex_by_prob", None)   # the answer you expect

In [ ]:
q = "How much will Flex 30 cost?"
rs = [bertlab.read(q, n) for n in notices]
for i, r in enumerate(rs):
    print(i, f"prob {r['prob']:.3f}  logit {r['logit']:6.2f}  {r['answer']!r}")
results["flex_by_prob"] = max(rs, key=lambda r: r["prob"])["answer"]
results["flex_by_logit"] = max(rs, key=lambda r: r["logit"])["answer"]
print("by prob:", results["flex_by_prob"], "| by logit:", results["flex_by_logit"])
reveal("flex_by_prob", results["flex_by_prob"])

By `prob`, `20 seconds`, from the Nimbus X2 notice, which is not a price at all. `prob` is a softmax **over
one notice**: it says how much the reader prefers this span to the other spans in the same notice, so in a
notice with one plausible-looking number it is high whether or not the notice is about Flex 30. It cannot
be compared between notices.

The raw `logit` can, and it does better: `$5.00`, at least a price. But it is the outage credit, from the
wrong notice. The reader judges whether a span looks like the answer to "how much", and a price in any
notice does. It was never trained to decide which document is about Flex 30, because SQuAD always handed it
the right paragraph.

## 5. Your turn: find the notice first

So give it the right paragraph. Search engines have ranked documents against queries for decades, and
Lab 02 gave you the classic tool: TF-IDF and cosine similarity. The cell builds a TF-IDF index of the five
notices and scores the question against each. Fill in the line that picks the notice with the highest
score; the reader then reads only that one. This pairing, a **retriever** and a **reader**, is how
question answering over documents is built, and Lab 12 builds on it.

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
vec = TfidfVectorizer(stop_words="english").fit(notices)
scores = (vec.transform([q]) @ vec.transform(notices).T).toarray()[0]
print("TF-IDF score of each notice:", scores.round(2))

best_notice = None   # YOUR CODE HERE: the index of the highest score (scores.argmax())

results["flex_by_retrieval"] = bertlab.read(q, notices[best_notice])["answer"] if best_notice is not None else None
print("retrieve, then read:", results["flex_by_retrieval"])
with open("out/11_03_results.json", "w") as f:
    json.dump(results, f, indent=1)
check_11_03()

## 6. How sure is the reader when it should not answer?

The logit is also your best signal for when to decline. Here are the highest logits across all five notices
for questions the notices answer, and for questions they do not. Look at where the two groups sit before you
start Part 3.

In [ ]:
for q in ["When will the mast be upgraded?", "What caused the outage?", "Where are the new 5G masts?",
          "How much is Unlimited Plus?", "Does the Corvid 5 have a headphone jack?", "Who founded Kittiwake Mobile?"]:
    rs = [bertlab.read(q, n) for n in notices]
    best = max(rs, key=lambda r: r["logit"])
    print(f"{best['logit']:6.2f}  {q:45s} -> {best['answer']!r}")

The answered questions score high, and so do some that the notices cannot answer: nobody says who founded
Kittiwake, but the pricing notice contains a person's name, and a name is what a "who" question wants. Read
across all five notices, the reader is at its most confident exactly where it is wrong. Reading only the
notice the retriever picked, and declining when even that reading scores low, is the answer desk Part 3 asks
you to build.

## 7. Exit ticket

In one or two sentences: why can a softmax probability not tell you which of five notices holds the answer,
and what did the retriever add?

*Your answer here.*